# 04. Aplicar Splink 5 no Spark — Luis nacional

Predict com `splink_model_mae_endereco_luis.json` nos CSVs de entrega do notebook 07.
**Spark-only**.

Entrada em `/data/spark/shared/datasets/`:
- `censo_aplicacao_splink.csv` / `cpf_aplicacao_splink.csv` (sem header)
- `splink_model_mae_endereco_luis.json`

Saída: `splink_predictions_mae_endereco_luis.parquet`.


In [ ]:
import os
from pathlib import Path

import json

import splink
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType
from splink.backends.spark import SparkAPI

# 'small' ou 'full' — ver spark_cluster.json
CLUSTER = 'small'

_cfg_candidates = [
    Path.cwd() / 'spark_cluster.json',
    Path.cwd() / 'spark' / 'spark_cluster.json',
    Path.cwd() / 'notebooks' / 'spark' / 'spark_cluster.json',
]
CFG_PATH = next(p for p in _cfg_candidates if p.exists())
cfg_all = json.loads(CFG_PATH.read_text())
profile = cfg_all['profiles'][CLUSTER]

os.environ['JAVA_HOME'] = cfg_all['java_home']
os.environ['SPARK_HOME'] = cfg_all['spark_home']
os.environ['PYSPARK_PYTHON'] = cfg_all['python']
os.environ['PYSPARK_DRIVER_PYTHON'] = cfg_all['python']

jar_path = cfg_all['jar']
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    f'--jars {jar_path} '
    f'--driver-class-path {jar_path} '
    'pyspark-shell'
)

builder = (
    SparkSession.builder
    .appName('04_aplicar_luis_nacional')
    .master(cfg_all['master'])
    .config('spark.driver.host', cfg_all['driver']['host'])
    .config('spark.driver.bindAddress', cfg_all['driver']['bindAddress'])
    .config('spark.driver.port', cfg_all['driver']['port'])
    .config('spark.pyspark.python', cfg_all['python'])
)
for key, value in profile.items():
    builder = builder.config(key, value)

spark = builder.getOrCreate()

DATA_DIR = Path('/data/spark/shared/datasets')
CKPT_DIR = cfg_all['checkpoint_dir']
spark.sparkContext.setCheckpointDir(CKPT_DIR)
spark.sparkContext.setLogLevel('ERROR')
ckpt = spark.sparkContext.getCheckpointDir()
if not ckpt:
    raise RuntimeError('Checkpoint dir nao configurado no SparkContext')

print('Spark:', spark.version)
print('Master:', spark.sparkContext.master)
print('App:', spark.sparkContext.applicationId)
print('cluster profile:', CLUSTER)
print('splink:', getattr(splink, '__version__', '?'))
print('jar:', jar_path)
print('DATA_DIR:', DATA_DIR)
print('checkpoint:', ckpt)

CENSO_CSV = DATA_DIR / 'censo_aplicacao_splink.csv'
CPF_CSV = DATA_DIR / 'cpf_aplicacao_splink.csv'
MODEL_JSON = DATA_DIR / 'splink_model_mae_endereco_luis.json'
OUT_PARQUET = DATA_DIR / 'splink_predictions_mae_endereco_luis.parquet'

COLS = [
    'id_seq',
    'nome_completo_phon',
    'primeiro_nome_phon',
    'ultimo_nome_phon',
    'data_nascimento',
    'ano_nascimento',
    'mes_nascimento',
    'dia_nascimento',
    'idade',
    'nome_mae_phon',
    'logradouro_norm',
    'cep',
    'cod_municipio',
    'sexo',
]
schema = StructType([StructField(c, StringType(), True) for c in COLS])

print('modelo:', MODEL_JSON)
print('censo:', CENSO_CSV)
print('cpf:  ', CPF_CSV)
print('saida:', OUT_PARQUET)


In [ ]:
df_censo = spark.read.csv(str(CENSO_CSV), header=False, schema=schema)
df_censo = df_censo.withColumn(
    'unique_id', F.concat(F.lit('censo_'), F.col('id_seq'))
)
df_censo = df_censo.withColumn('idade', F.col('idade').cast(IntegerType()))
df_censo = df_censo.withColumn('source_dataset', F.lit('censo'))

df_cpf = spark.read.csv(str(CPF_CSV), header=False, schema=schema)
df_cpf = df_cpf.withColumn(
    'unique_id', F.concat(F.lit('cpf_'), F.col('id_seq'))
)
df_cpf = df_cpf.withColumn('idade', F.col('idade').cast(IntegerType()))
df_cpf = df_cpf.withColumn('source_dataset', F.lit('cpf'))

print('censo:', df_censo.count())
print('cpf:  ', df_cpf.count())
df_censo.select(
    'unique_id', 'nome_completo_phon', 'data_nascimento', 'idade'
).show(3, truncate=False)


In [ ]:
from splink import block_on

blocking_rules = [
    block_on('nome_completo_phon'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'data_nascimento'),
    block_on('ultimo_nome_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'dia_nascimento', 'cod_municipio'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'ano_nascimento', 'cod_municipio'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento', 'sexo', 'cep'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento', 'sexo', 'cep'),
    (
        "l.ultimo_nome_phon = r.ultimo_nome_phon "
        "AND l.ano_nascimento = r.ano_nascimento "
        "AND l.mes_nascimento = r.mes_nascimento "
        "AND l.sexo = r.sexo "
        "AND damerau_levenshtein(l.primeiro_nome_phon, r.primeiro_nome_phon) <= 2 "
        "AND least(len(l.primeiro_nome_phon), len(r.primeiro_nome_phon)) >= 6 "
        "AND damerau_levenshtein(l.primeiro_nome_phon, r.primeiro_nome_phon) * 6 "
        "<= least(len(l.primeiro_nome_phon), len(r.primeiro_nome_phon))"
    ),
    (
        "l.primeiro_nome_phon = r.primeiro_nome_phon "
        "AND l.ultimo_nome_phon = r.ultimo_nome_phon "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
    (
        "l.logradouro_norm = r.logradouro_norm "
        "AND l.cep = r.cep "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
    (
        "l.nome_mae_phon = r.nome_mae_phon "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
]


In [ ]:
import inspect

from splink import Linker, SettingsCreator

with open(MODEL_JSON) as f:
    settings = json.load(f)

settings['retain_intermediate_calculation_columns'] = False
settings['retain_matching_columns'] = False
settings['blocking_rules_to_generate_predictions'] = blocking_rules
settings.pop('sql_dialect', None)

allowed = set(inspect.signature(SettingsCreator.__init__).parameters) - {'self'}
drop = sorted(k for k in settings if k not in allowed)
if drop:
    print('ignorando chaves do JSON:', drop)
settings = {k: v for k, v in settings.items() if k in allowed}

# Splink 5: register + Linker([SplinkDataFrame, ...], settings)
spark.sparkContext.setCheckpointDir(CKPT_DIR)
db_api = SparkAPI(
    spark_session=spark,
    break_lineage_method='checkpoint',
    repartition_after_blocking=False,
)
censo_in = db_api.register(df_censo, dataset_display_name='censo')
cpf_in = db_api.register(df_cpf, dataset_display_name='cpf')
linker = Linker([censo_in, cpf_in], settings)

print('Linker pronto:', MODEL_JSON)


In [ ]:
df_predict = linker.inference.predict(threshold_match_probability=0.05)
pred = df_predict.as_spark_dataframe()
print('predictions:', pred.count())
pred.select(
    'unique_id_l', 'unique_id_r', 'match_probability', 'match_weight'
).show(5)

keep = [
    c for c in (
        'unique_id_l', 'unique_id_r', 'match_probability', 'match_weight'
    )
    if c in pred.columns
]

(
    pred.select(*keep)
    .write.mode('overwrite')
    .parquet(str(OUT_PARQUET))
)

print('Predictions:', OUT_PARQUET)
print('colunas:', keep)
